# Практика. Локализации, метод `apply` и конкатенации

In [1]:
import pandas as pd

>#### Задание 1
Прочитайте файлы `retail_north.csv` и `retail_south.csv`.

In [2]:
df_N = pd.read_csv('data/retail_north.csv')
df_S = pd.read_csv('data/retail_south.csv')
display(df_N.head(3))
display(df_S.head(3))

,Model_ID,Units_Sold_N,Revenue_N,Returns_N,Price_N
0,SM0084,359,4835868,16,63441
1,SM0275,304,4272811,12,38865
2,SM0244,162,1333727,26,56240


,Model_ID,Units_Sold_S,Revenue_S,Returns_S,Price_S
0,SM0023,149,476076,22,14358
1,SM0110,266,3003381,15,10207
2,SM0029,177,545546,15,52533


#### Контекст

Сеть магазинов электроники анализирует продажи смартфонов в двух разных регионах (Север и Юг), чтобы решить, какие модели оставить, а какие вывести из розницы.

`df_north` и `df_south` — таблицы продаж. Столбцы одинаковые: `Model_ID` (ID серии), `Units_Sold` (количество единиц), `Revenue` (выручка), `Returns` (возвраты), `Price` (средняя цена).

>#### Задание 2
Есть ли в списках для севера и юга общие модели? Есть ли уникальные уникальные для каждого региона? Выведите списки общих и уникальных моделей. Должно получиться три списка:
>- модели, общие для обоих регионов,
>- модели, которые продаются только на севере,
>- модели, которые продаются только на юге.

In [5]:
N = set(df_N['Model_ID'])
S = set(df_S['Model_ID'])
N_and_S = N & S
N_and_S  = N - S
S_only  = S - N
S_only
S_only

set()

>#### Задание 3
Объедините таблицы методом `concat` так, чтобы получить общую таблицу по всем моделям в обоих регионах (включая те модели, которые продавались только в одном месте — при этом возникнут NaN’ы).
>
>Подсказка: в качестве индекса нужно использовать ID моделей. А после завершения конкатенации индекс нужно сбросить при помощи метода `reset_index` (следует указать drop=True, так как индексные столбец `Model_ID` уже есть в датафрейме, и `reset_index` не сможет его встваить еще раз, сообщив об ошибке).

In [6]:
df_N.index = df_N['Model_ID']
df_S.index = df_S['Model_ID']
display(df_N.head(3))
display(df_S.head(3))

,Model_ID,Units_Sold_N,Revenue_N,Returns_N,Price_N
Model_ID,,,,,
SM0084,SM0084,359,4835868,16,63441
SM0275,SM0275,304,4272811,12,38865
SM0244,SM0244,162,1333727,26,56240


,Model_ID,Units_Sold_S,Revenue_S,Returns_S,Price_S
Model_ID,,,,,
SM0023,SM0023,149,476076,22,14358
SM0110,SM0110,266,3003381,15,10207
SM0029,SM0029,177,545546,15,52533


In [7]:
combine = pd.concat([df_N, df_S], axis=1)
combine

,Model_ID,Units_Sold_N,Revenue_N,Returns_N,Price_N,Model_ID,Units_Sold_S,Revenue_S,Returns_S,Price_S
Model_ID,,,,,,,,,,
SM0084,SM0084,359,4835868,16,63441,SM0084,245.0,398818.0,1.0,56479.0
SM0275,SM0275,304,4272811,12,38865,SM0275,314.0,1643862.0,24.0,33291.0
SM0244,SM0244,162,1333727,26,56240,SM0244,121.0,861165.0,0.0,65922.0
SM0099,SM0099,345,2303649,7,11917,SM0099,262.0,601207.0,9.0,41055.0
SM0062,SM0062,395,2957184,5,58447,SM0062,108.0,3326217.0,15.0,60331.0
...,...,...,...,...,...,...,...,...,...,...
SM0049,SM0049,180,1517026,0,57915,SM0049,68.0,3057160.0,18.0,64823.0
SM0256,SM0256,273,4241196,9,31566,NaN,NaN,NaN,NaN,NaN
SM0136,SM0136,250,2558852,18,30611,SM0136,256.0,568512.0,10.0,17421.0


In [8]:
combine = combine.reset_index(drop=True)
combine

,Model_ID,Units_Sold_N,Revenue_N,Returns_N,Price_N,Model_ID,Units_Sold_S,Revenue_S,Returns_S,Price_S
0,SM0084,359,4835868,16,63441,SM0084,245.0,398818.0,1.0,56479.0
1,SM0275,304,4272811,12,38865,SM0275,314.0,1643862.0,24.0,33291.0
2,SM0244,162,1333727,26,56240,SM0244,121.0,861165.0,0.0,65922.0
3,SM0099,345,2303649,7,11917,SM0099,262.0,601207.0,9.0,41055.0
4,SM0062,395,2957184,5,58447,SM0062,108.0,3326217.0,15.0,60331.0
...,...,...,...,...,...,...,...,...,...,...
295,SM0049,180,1517026,0,57915,SM0049,68.0,3057160.0,18.0,64823.0
296,SM0256,273,4241196,9,31566,NaN,NaN,NaN,NaN,NaN
297,SM0136,250,2558852,18,30611,SM0136,256.0,568512.0,10.0,17421.0
298,SM0223,367,3264673,16,73374,NaN,NaN,NaN,NaN,NaN


#### Вопрос
Что не так? Правильно: ID модели используется дважды: сначала как столбец `Model_ID` (это пришло с севера), а потом еще раз как столбец `Model_ID` (это пришло с юга). 

>#### Задание 4
Удалите повторяющийся столбец при помощи метода `loc` за счет локализации по столбцам с использованием маски `~df.columns.duplicated()`.

In [9]:
combine = combine.loc[:, ~combine.columns.duplicated()]
combine

,Model_ID,Units_Sold_N,Revenue_N,Returns_N,Price_N,Units_Sold_S,Revenue_S,Returns_S,Price_S
0,SM0084,359,4835868,16,63441,245.0,398818.0,1.0,56479.0
1,SM0275,304,4272811,12,38865,314.0,1643862.0,24.0,33291.0
2,SM0244,162,1333727,26,56240,121.0,861165.0,0.0,65922.0
3,SM0099,345,2303649,7,11917,262.0,601207.0,9.0,41055.0
4,SM0062,395,2957184,5,58447,108.0,3326217.0,15.0,60331.0
...,...,...,...,...,...,...,...,...,...
295,SM0049,180,1517026,0,57915,68.0,3057160.0,18.0,64823.0
296,SM0256,273,4241196,9,31566,NaN,NaN,NaN,NaN
297,SM0136,250,2558852,18,30611,256.0,568512.0,10.0,17421.0
298,SM0223,367,3264673,16,73374,NaN,NaN,NaN,NaN


>#### Задание 5
Заполните пропуски нулями, используя знакомый вам по более ранним заданиям метод `fillna`, и обратите внимание, что целочисленные значения, которые стали дробными из-за NaN'ов, остались дробными. Исправьте эту досадную неточность при помощи метода `astype`. 
>
>Затем при помощи `apply` вычислите суммарную выручку для каждой модели и прикрепите ее к объединённому датафрейму в виде столбца `Total`.

In [10]:
combine = combine.fillna(0)
combine

,Model_ID,Units_Sold_N,Revenue_N,Returns_N,Price_N,Units_Sold_S,Revenue_S,Returns_S,Price_S
0,SM0084,359,4835868,16,63441,245.0,398818.0,1.0,56479.0
1,SM0275,304,4272811,12,38865,314.0,1643862.0,24.0,33291.0
2,SM0244,162,1333727,26,56240,121.0,861165.0,0.0,65922.0
3,SM0099,345,2303649,7,11917,262.0,601207.0,9.0,41055.0
4,SM0062,395,2957184,5,58447,108.0,3326217.0,15.0,60331.0
...,...,...,...,...,...,...,...,...,...
295,SM0049,180,1517026,0,57915,68.0,3057160.0,18.0,64823.0
296,SM0256,273,4241196,9,31566,0.0,0.0,0.0,0.0
297,SM0136,250,2558852,18,30611,256.0,568512.0,10.0,17421.0
298,SM0223,367,3264673,16,73374,0.0,0.0,0.0,0.0


In [11]:
combine[combine.columns[1:]] = combine[combine.columns[1:]].astype(int)
combine

,Model_ID,Units_Sold_N,Revenue_N,Returns_N,Price_N,Units_Sold_S,Revenue_S,Returns_S,Price_S
0,SM0084,359,4835868,16,63441,245,398818,1,56479
1,SM0275,304,4272811,12,38865,314,1643862,24,33291
2,SM0244,162,1333727,26,56240,121,861165,0,65922
3,SM0099,345,2303649,7,11917,262,601207,9,41055
4,SM0062,395,2957184,5,58447,108,3326217,15,60331
...,...,...,...,...,...,...,...,...,...
295,SM0049,180,1517026,0,57915,68,3057160,18,64823
296,SM0256,273,4241196,9,31566,0,0,0,0
297,SM0136,250,2558852,18,30611,256,568512,10,17421
298,SM0223,367,3264673,16,73374,0,0,0,0


In [12]:
def total(row):
    total = row['Units_Sold_N']*row['Price_N'] + row['Units_Sold_S']*row['Price_S']
    return total

combine['Total'] = combine.apply(total, axis=1)
combine

,Model_ID,Units_Sold_N,Revenue_N,Returns_N,Price_N,Units_Sold_S,Revenue_S,Returns_S,Price_S,Total
0,SM0084,359,4835868,16,63441,245,398818,1,56479,36612674
1,SM0275,304,4272811,12,38865,314,1643862,24,33291,22268334
2,SM0244,162,1333727,26,56240,121,861165,0,65922,17087442
3,SM0099,345,2303649,7,11917,262,601207,9,41055,14867775
4,SM0062,395,2957184,5,58447,108,3326217,15,60331,29602313
...,...,...,...,...,...,...,...,...,...,...
295,SM0049,180,1517026,0,57915,68,3057160,18,64823,14832664
296,SM0256,273,4241196,9,31566,0,0,0,0,8617518
297,SM0136,250,2558852,18,30611,256,568512,10,17421,12112526
298,SM0223,367,3264673,16,73374,0,0,0,0,26928258


>#### Задание 6
При помощи метода `apply` вычислите «коэффициент брака» (это отношение суммы возвратов к сумме проданных единиц), учитывая продажи в обоих регионах. Припишите этот коэффициент в виде еще одного столбца `Defect` к комбинированному датафрейму.

In [13]:
def defect(row):
    total_sold = row['Units_Sold_N'] + row['Units_Sold_S']
    total_returns = row['Returns_N'] + row['Returns_S']
    return round(total_returns/total_sold, 2)

combine['Defect'] = combine.apply(defect, axis=1)
combine

,Model_ID,Units_Sold_N,Revenue_N,Returns_N,Price_N,Units_Sold_S,Revenue_S,Returns_S,Price_S,Total,Defect
0,SM0084,359,4835868,16,63441,245,398818,1,56479,36612674,0.03
1,SM0275,304,4272811,12,38865,314,1643862,24,33291,22268334,0.06
2,SM0244,162,1333727,26,56240,121,861165,0,65922,17087442,0.09
3,SM0099,345,2303649,7,11917,262,601207,9,41055,14867775,0.03
4,SM0062,395,2957184,5,58447,108,3326217,15,60331,29602313,0.04
...,...,...,...,...,...,...,...,...,...,...,...
295,SM0049,180,1517026,0,57915,68,3057160,18,64823,14832664,0.07
296,SM0256,273,4241196,9,31566,0,0,0,0,8617518,0.03
297,SM0136,250,2558852,18,30611,256,568512,10,17421,12112526,0.06
298,SM0223,367,3264673,16,73374,0,0,0,0,26928258,0.04


>#### Задание 7
С помощью `loc` выведите модели, у которых коэффициент брака выше 10%, чтобы передать их в отдел контроля качества.

In [14]:
low_quality = combine.loc[combine['Defect'] > 0.1, ['Model_ID', 'Defect']]
low_quality

,Model_ID,Defect
11,SM0040,0.11
14,SM0008,0.20
19,SM0141,0.20
23,SM0052,0.19
27,SM0167,0.20
37,SM0079,0.37
41,SM0133,0.32
43,SM0007,0.12
55,SM0195,0.11
64,SM0134,0.12


# Домашнее задание

>#### Задание 1
Прочитайте файлы `health_start.csv` и `health_final.csv`.

In [21]:
df_start = pd.read_csv('data/health_start.csv')
df_final = pd.read_csv('data/health_final.csv')
display(df_start.head(3))
display(df_final.head(3))

,Patient_ID,Hemoglobin,Vitamin_D,Cortisol
0,P0011,119.8,32.8,10.7
1,P0481,115.6,31.6,16.1
2,P0317,126.3,24.6,8.3


,Patient_ID,Hemoglobin,Vitamin_D,Cortisol
0,P0221,166.0,48.8,11.3
1,P0025,142.6,43.5,13.2
2,P0309,129.6,48.3,12.5


#### Контекст

В клинике проводят обследование группы пациентов до курса витаминной терапии и после.

`health_start` — показатели до, `health_final` — показатели через 3 месяца. Столбцы одинаковые: `Patient_ID` (ID пациента), `Hemoglobin` (гемоглобин), `Vitamin_D` (витамин D), `Cortisol` (костирол, гормон стресса). Пациенты в таблицах перемешаны, а некоторые не стали проходить финальное обследование.

>#### Задание 2
Переиндексируйте таблицы по `Patient_ID` и конкатенируйте их слева направо, используя `axis=1`.

In [22]:
# Переименовываем столбцы во втором датафрейме, чтобы не было путаницы после конкатенации
columns_2 = [x + '_2' for x in df_final.columns]
columns_2

['Patient_ID_2', 'Hemoglobin_2', 'Vitamin_D_2', 'Cortisol_2']

In [17]:
# Заводим копию финальных измерений, исходник с родными именами столбцов мне еще понадобится
df_final_2 = df_final.copy()

# Переименовываем столбцы в копии
df_final_2.columns = columns_2

# Заменяем индексы на ID
df_start.index = df_start['Patient_ID']
df_final_2.index = df_final_2['Patient_ID_2']

# Конкатенируем слева направо
combine = pd.concat([df_start, df_final_2], axis=1)

# Сбасываем индекс (терпеть не могу нестандартные индексы)
combine = combine.reset_index(drop=True)
combine

,Patient_ID,Hemoglobin,Vitamin_D,Cortisol,Patient_ID_2,Hemoglobin_2,Vitamin_D_2,Cortisol_2
0,P0011,119.8,32.8,10.7,P0011,130.7,42.9,6.7
1,P0481,115.6,31.6,16.1,P0481,137.9,52.9,11.3
2,P0317,126.3,24.6,8.3,P0317,126.9,47.8,15.3
3,P0372,121.2,14.3,17.9,P0372,154.9,56.0,7.4
4,P0397,125.6,40.7,23.2,P0397,145.9,43.5,10.1
...,...,...,...,...,...,...,...,...
595,P0484,148.9,32.9,21.1,P0484,145.3,37.9,12.0
596,P0381,141.8,27.3,0.3,P0381,135.8,46.2,11.1
597,P0566,149.6,33.2,17.7,P0566,111.4,44.7,10.6
598,P0068,126.5,46.4,18.3,P0068,150.7,57.9,11.8


>#### Задание 3
Напишите функцию `get_status`, которая анализирует динамику показателей пациента (до и после терапии) и возвращает один из четырех статусов:
>- «Неизвестно» — если пациент не стал проходить второе обследование;
>- «Улучшение» — если одновременно: `Vitamin_D` вырос более чем на 20%, `Cortisol` снизился, `Hemoglobin` вырос более чем на 5%;
>- «Требуется врач» — если хотя бы один из показателей значительно ухудшился: `Vitamin_D` упал более чем на 15%, или `Cortisol` вырос более чем на 20%, или `Hemoglobin` упал более чем на 10%;
>- «Без изменений» — все остальные случаи (динамика незначительная или противоречивая).
>
>Оформите это в новый столбец `Report` через `apply`.

In [18]:
def get_status(row):
    
    # Условия для улучшения
    A1 = row['Vitamin_D_2'] > 1.2*row['Vitamin_D']
    A2 = row['Cortisol_2'] < row['Cortisol']
    A3 = row['Hemoglobin_2'] > 1.05*row['Hemoglobin']
    
    # Условия для ухудшения (требуется врач)
    B1 = row['Vitamin_D_2'] < 0.85*row['Vitamin_D']
    B2 = row['Cortisol_2'] > 1.2*row['Cortisol']
    B3 = row['Hemoglobin_2'] < 0.9*row['Hemoglobin']
    
    # Выписываем логику медицинского заключения
    if row['Vitamin_D_2'] != row['Vitamin_D_2']:
        return 'Неизвестно'    
    elif A1 & A2 & A3:
        return 'Улучшение'
    elif B1 | B2 | B3:
        return 'Требуется врач'
    else:
        return 'Без изменений'

# Добавляем столбец
combine['Report'] = combine.apply(get_status, axis=1)
combine

,Patient_ID,Hemoglobin,Vitamin_D,Cortisol,Patient_ID_2,Hemoglobin_2,Vitamin_D_2,Cortisol_2,Report
0,P0011,119.8,32.8,10.7,P0011,130.7,42.9,6.7,Улучшение
1,P0481,115.6,31.6,16.1,P0481,137.9,52.9,11.3,Улучшение
2,P0317,126.3,24.6,8.3,P0317,126.9,47.8,15.3,Требуется врач
3,P0372,121.2,14.3,17.9,P0372,154.9,56.0,7.4,Улучшение
4,P0397,125.6,40.7,23.2,P0397,145.9,43.5,10.1,Без изменений
...,...,...,...,...,...,...,...,...,...
595,P0484,148.9,32.9,21.1,P0484,145.3,37.9,12.0,Без изменений
596,P0381,141.8,27.3,0.3,P0381,135.8,46.2,11.1,Требуется врач
597,P0566,149.6,33.2,17.7,P0566,111.4,44.7,10.6,Требуется врач
598,P0068,126.5,46.4,18.3,P0068,150.7,57.9,11.8,Улучшение


>#### Задание 4
Используя `loc`, посчитайте, сколько пациентов получили статус «Улучшение».

In [19]:
len(combine[combine['Report'] == 'Улучшение'])

133

>#### Задание 5
Конкатенируйте исходные датафреймы сверху вниз при помощи `concat(axis=0)`, после чего, используя `apply(axis=0)`, найдите сводную статистику: среднее изменение по каждому показателю для всей клиники.

In [20]:
# Сбрасываем индексы в исходниках
df_start = df_start.reset_index(drop=True)
df_final = df_final.reset_index(drop=True)

# Конкатенируем сверху вниз, axis=0 по дефолту, поэтому не указываем
combine = pd.concat([df_start, df_final])

# Удаляем ID, они нам не понадобятся
combine = combine.drop(columns = ['Patient_ID'])

# Заводим внешнюю функцию с логикой сводной статистики
def summary_statistics(column):
    start = column[:len(df_start)].mean()
    final = column[len(df_start):].mean()
    delta = final - start
    return round(delta, 2)

# Применяем apply по столбцам, axis=0 по дефолту, поэтому не указываем
summary =  combine.apply(summary_statistics) 
summary

Hemoglobin     5.25
Vitamin_D     15.94
Cortisol      -2.94
dtype: float64